# Fundamentos de LLMs con Hugging Face + introducción a LangChain

En este notebook vamos a aprender a **cargar, configurar, usar y adaptar** modelos de lenguaje pequeños con las librerías de Hugging Face (`transformers`, `datasets`, `peft`) y terminaremos con una **introducción práctica a LangChain**.

| # | Sección | Modelos |
|---|---------|---------|
| 0 | Preparación del entorno | – |
| 1 | Las tres familias de Transformers: qué sabe hacer cada una | [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased), [DistilBERT-SST2](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english), [BERT-NER](https://huggingface.co/dslim/bert-base-NER), [MiniLM](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), [SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), [Flan-T5-small](https://huggingface.co/google/flan-t5-small) |
| 2 | Adaptación al dominio (recetas de cocina) | [SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), [Flan-T5-small](https://huggingface.co/google/flan-t5-small), [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) |
| 3 | Parámetros de generación y *chat templates* | [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) |
| 4 | Modelo con *instruction tuning* vs. modelo sin él | [T5-base](https://huggingface.co/google-t5/t5-base) vs [Flan-T5-base](https://huggingface.co/google/flan-t5-base), [SmolLM2-360M](https://huggingface.co/HuggingFaceTB/SmolLM2-360M) vs [SmolLM2-360M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct) |
| 5 | Introducción a LangChain | LLM vía OpenRouter + [MiniLM](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) (embeddings) |

> **Nota:** todos los modelos son **pequeños** (≤ 0.5B parámetros): funcionan en una GPU de ~6 GB o en Google Colab (GPU T4). En CPU también funcionan, pero el entrenamiento de la sección 2 será lento: activa `MODO_RAPIDO`.

Al final de cada sección hay una celda **Ejercicios**.

---
## 0. Preparación del entorno

Si estás en **Google Colab**, la siguiente celda instala las librerías necesarias. En local (entorno creado con `uv`, ver `README.md`) no hace nada.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q -U transformers datasets accelerate peft evaluate sentencepiece sentence-transformers langchain langchain-openai langchain-huggingface python-dotenv
print("¿Estamos en Colab?", IN_COLAB)

In [ ]:
import gc
import math
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import logging as hf_logging

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
pd.set_option("display.max_colwidth", None)

# Dispositivo: GPU NVIDIA (cuda), Apple Silicon (mps) o CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# bf16 solo en GPUs modernas (Ampere o superior: RTX 30xx/40xx, A100...). La T4 de Colab usa fp16.
USE_BF16 = device == "cuda" and torch.cuda.is_bf16_supported()
USE_FP16 = device == "cuda" and not USE_BF16
DTYPE_INFERENCIA = torch.bfloat16 if USE_BF16 else (torch.float16 if device == "cuda" else torch.float32)

SEED = 42
transformers.set_seed(SEED)

print(f"transformers {transformers.__version__} | torch {torch.__version__}")
print(f"Dispositivo: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""))
print(f"bf16: {USE_BF16} | fp16: {USE_FP16}")

Algunas **funciones auxiliares** que reutilizaremos en todo el notebook:

In [ ]:
def contar_parametros(model):
    # Número de parámetros del modelo en millones
    return f"{model.num_parameters() / 1e6:.0f}M parámetros"


def liberar_memoria():
    # Libera la memoria de la GPU (llámala después de hacer `del model`)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"VRAM ocupada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


@torch.no_grad()
def generar(model, tok, prompt, **gen_kwargs):
    # Genera texto con un modelo decoder-only y devuelve SOLO el texto nuevo
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    gen_kwargs.setdefault("max_new_tokens", 60)
    gen_kwargs.setdefault("pad_token_id", tok.pad_token_id or tok.eos_token_id)
    out = model.generate(**inputs, **gen_kwargs)
    nuevos = out[0, inputs["input_ids"].shape[1]:]
    return tok.decode(nuevos, skip_special_tokens=True)


@torch.no_grad()
def generar_seq2seq(model, tok, prompt, **gen_kwargs):
    # Genera texto con un modelo encoder-decoder (T5)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    gen_kwargs.setdefault("max_new_tokens", 60)
    out = model.generate(**inputs, **gen_kwargs)
    return tok.decode(out[0], skip_special_tokens=True)

---
## 1. Las tres familias de Transformers: qué sabe hacer cada una

Todos los LLMs actuales son **Transformers**, pero según qué partes de la arquitectura original usen, se dividen en tres familias:

| Familia | Atención | Objetivo de preentrenamiento | Se le da bien | Ejemplos |
|---|---|---|---|---|
| **Encoder** | Bidireccional (ve toda la frase) | *Masked LM*: adivinar palabras tapadas `[MASK]` | **Entender**: clasificar, extraer entidades, embeddings, búsqueda semántica | BERT, DistilBERT, RoBERTa, MiniLM |
| **Decoder** (*decoder-only*) | Causal (solo ve lo anterior) | *Causal LM*: predecir el siguiente token | **Generar** texto, chat, código | GPT, Llama, Qwen, SmolLM |
| **Encoder-decoder** (*seq2seq*) | Encoder bidireccional + decoder causal | Reconstruir/transformar texto (*span corruption*) | **Transformar** un texto en otro: traducir, resumir, responder | T5, Flan-T5, BART, mT5 |

En Hugging Face hay dos formas de usar un modelo:

1. **`pipeline(...)`**: alto nivel, en una línea.
2. **`AutoTokenizer` + `AutoModelFor...`**: bajo nivel, controlamos cada paso (tokenizar, pasar por el modelo, interpretar la salida).

Vamos a verlas las dos.

### 1.1 Encoder: comprender texto

#### a) *Fill-mask*: la tarea con la que se preentrenó BERT

Modelo: [`distilbert-base-uncased`](https://huggingface.co/distilbert/distilbert-base-uncased)

In [ ]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model="distilbert-base-uncased", device=device)

for pred in fill_mask("The chef put the cake in the [MASK] for thirty minutes.", top_k=5):
    print(f"{pred['score']:.3f}  {pred['token_str']}")

#### b) Clasificación de texto (análisis de sentimiento)

El mismo DistilBERT, pero **ajustado** (*fine-tuned*) para clasificar sentimiento (dataset SST-2). Se ha añadido una "cabeza" de clasificación encima del encoder.

Modelo: [`distilbert-base-uncased-finetuned-sst-2-english`](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english)

In [ ]:
sentiment = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english", device=device)

reviews = [
    "The paella was absolutely delicious, I will come back!",
    "The soup was cold and the waiter was rude.",
    "It was ok, nothing special.",
]
for r, pred in zip(reviews, sentiment(reviews)):
    print(f"{pred['label']:8s} {pred['score']:.3f}  | {r}")

#### c) Reconocimiento de entidades (NER): clasificación a nivel de token

Modelo: [`dslim/bert-base-NER`](https://huggingface.co/dslim/bert-base-NER) (BERT ajustado para detectar personas, lugares, organizaciones y otras entidades)

In [ ]:
ner = pipeline("token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple", device=device)

for ent in ner("Gordon Ramsay opened a new restaurant in London with Jamie Oliver, near the Starbucks of Oxford Street."):
    print(f"{ent['entity_group']:5s} {ent['score']:.3f}  {ent['word']}")

#### d) Embeddings y similitud semántica

Un encoder también sirve para convertir un texto en un **vector** (*embedding*). Textos con significado parecido tienen vectores cercanos. Esto es la base de los buscadores semánticos y del **RAG** (lo usaremos en la sección 5).

Modelo: [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

recetas = [
    "Chocolate chip cookies",
    "Spaghetti carbonara",
    "Vanilla sponge cake",
    "Grilled chicken salad",
    "Beef lasagna",
]
consulta = "a sweet dessert to bake"

emb_recetas = embedder.encode(recetas)
emb_consulta = embedder.encode([consulta])
print("Dimensión del embedding:", emb_recetas.shape)

sims = embedder.similarity(emb_consulta, emb_recetas)[0]
for receta, s in sorted(zip(recetas, sims.tolist()), key=lambda x: -x[1]):
    print(f"{s:.3f}  {receta}")

#### e) ¿Qué hay dentro de un `pipeline`?

Repetimos el análisis de sentimiento **paso a paso**: tokenizador → modelo → *logits* → probabilidades.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

nombre = "distilbert-base-uncased-finetuned-sst-2-english"
tok_cls = AutoTokenizer.from_pretrained(nombre)
model_cls = AutoModelForSequenceClassification.from_pretrained(nombre).to(device)
print(contar_parametros(model_cls))

texto = "The paella was absolutely delicious!"

# 1) Tokenizar: texto -> tokens -> ids
enc = tok_cls(texto, return_tensors="pt").to(device)
print("Tokens:", tok_cls.convert_ids_to_tokens(enc["input_ids"][0]))
print("IDs:   ", enc["input_ids"][0].tolist())

# 2) Pasar por el modelo -> logits (una puntuación por clase)
with torch.no_grad():
    logits = model_cls(**enc).logits
print("Logits:", logits)

# 3) Softmax -> probabilidades
probs = logits.softmax(dim=-1)[0]
for i, p in enumerate(probs):
    print(f"{model_cls.config.id2label[i]:8s} {p:.4f}")

### 1.2 Decoder-only: generar texto

Un decoder-only hace una sola cosa: **predecir el siguiente token**. Repitiendo esto en bucle, genera texto. Usaremos [`SmolLM2-135M`](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), un modelo **base** (solo preentrenado, sin ajustar para seguir instrucciones).

In [ ]:
from transformers import AutoModelForCausalLM

nombre = "HuggingFaceTB/SmolLM2-135M"
tok_dec = AutoTokenizer.from_pretrained(nombre)
model_dec = AutoModelForCausalLM.from_pretrained(nombre).to(device)
print(contar_parametros(model_dec))

# ¿Cuál es el siguiente token más probable?
prompt = "To make a good omelette you need"
enc = tok_dec(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    logits = model_dec(**enc).logits      # [batch, n_tokens, tamaño_vocabulario]
print("Forma de los logits:", tuple(logits.shape))

probs_siguiente = logits[0, -1].softmax(dim=-1)  # distribución del token que va DESPUÉS del último
top = probs_siguiente.topk(8)
for p, idx in zip(top.values, top.indices):
    print(f"{p:.3f}  {tok_dec.decode(idx)!r}")

In [ ]:
# Repitiendo la predicción token a token -> generación
print(prompt + generar(model_dec, tok_dec, prompt, max_new_tokens=40, do_sample=False))

Un modelo **base** no "obedece": simplemente **continúa** el texto como si fuera un documento de internet. Fíjate en lo que hace si le damos una instrucción:

In [ ]:
for p in ["Translate to French: I love cheese.", "Question: What is the capital of Spain?\nAnswer:"]:
    print(repr(p), "->", repr(generar(model_dec, tok_dec, p, max_new_tokens=30, do_sample=False)))
    print()

### 1.3 Encoder-decoder: transformar un texto en otro

T5 plantea **todas** las tareas como *texto → texto*. El **encoder** lee la entrada completa y el **decoder** genera la salida token a token, consultando al encoder (*cross-attention*).

Modelo: [`google/flan-t5-small`](https://huggingface.co/google/flan-t5-small)

> **Nota:** en `transformers` v5 ya no existen los pipelines `text2text-generation`, `summarization` ni `translation`, así que usaremos directamente `model.generate()` (función `generar_seq2seq` definida arriba).

In [ ]:
from transformers import AutoModelForSeq2SeqLM

nombre = "google/flan-t5-small"
tok_t5 = AutoTokenizer.from_pretrained(nombre)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(nombre).to(device)
print(contar_parametros(model_t5))

tareas = [
    "translate English to German: The kitchen is clean.",
    "summarize: Paella is a rice dish originally from Valencia. It is cooked in a wide, shallow pan "
    "and traditionally includes chicken, rabbit, green beans and saffron. Today there are many variants, "
    "such as seafood paella.",
    "Answer the question: What is the capital of France?",
    "Is this review positive or negative? Review: The soup was cold and tasteless.",
]
for t in tareas:
    print(f"> {t[:70]}...\n  {generar_seq2seq(model_t5, tok_t5, t)}\n")

> Con solo 77M de parámetros, Flan-T5-small acierta algunas tareas y falla otras de forma llamativa. Es normal: **los modelos pequeños se equivocan mucho**, sobre todo en tareas que requieren conocimiento del mundo.

In [ ]:
# Por dentro: el encoder produce UN vector por cada token de entrada, que el decoder consulta al generar
enc = tok_t5("translate English to German: The kitchen is clean.", return_tensors="pt").to(device)
with torch.no_grad():
    salida_encoder = model_t5.get_encoder()(**enc).last_hidden_state
print("Tokens de entrada:", len(enc["input_ids"][0]))
print("Salida del encoder:", tuple(salida_encoder.shape), "-> [batch, tokens, dimensión oculta]")

In [ ]:
# Liberamos memoria antes de la siguiente sección
del fill_mask, sentiment, ner, embedder, model_cls, model_dec, model_t5
liberar_memoria()

### Ejercicios — Sección 1

1. Prueba el *fill-mask* con 3 frases tuyas de cocina. ¿Acierta? ¿Qué pasa si pones dos `[MASK]`?
2. Busca en el [Hub de Hugging Face](https://huggingface.co/models) un modelo encoder de **análisis de sentimiento en español** y úsalo con `pipeline` con reseñas en castellano.
3. Añade 5 recetas más a la lista del apartado 1.1.d y haz 3 consultas distintas. ¿Funciona la búsqueda semántica aunque no coincidan las palabras?
4. Compara el número de parámetros de los modelos vistos (`contar_parametros`). ¿Cuál es el más grande?
5. Para cada uno de estos casos, ¿qué familia (encoder, decoder o encoder-decoder) usarías y por qué?
   - a) Detectar correos de *spam*.
   - b) Un chatbot.
   - c) Traducir manuales técnicos.
   - d) Buscar documentos parecidos a una consulta.
   - e) Autocompletar código.

---
## 2. Adaptación al dominio: recetas de cocina

Los modelos preentrenados saben "un poco de todo". **Adaptarlos al dominio** consiste en seguir entrenándolos con texto de nuestro dominio para que aprendan su vocabulario, su estilo y su estructura.

Todos usaremos el mismo dataset: [`corbt/all-recipes`](https://huggingface.co/datasets/corbt/all-recipes), con más de 2 millones de recetas en inglés y este formato:

```
Título

Ingredients:
- ...

Directions:
- ...
```

Vamos a adaptar **tres modelos**, cada uno con su objetivo de entrenamiento natural:

| Modelo | Familia | Tarea de entrenamiento | Lo que esperamos que aprenda |
|---|---|---|---|
| [SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M) | Decoder | *Causal LM*: continuar texto | Escribir recetas con el formato correcto |
| [Flan-T5-small](https://huggingface.co/google/flan-t5-small) | Encoder-decoder | *Seq2seq*: título → receta | Generar ingredientes y pasos a partir de un título |
| [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) | Encoder | *Masked LM* (opcional) | Vocabulario de cocina |

### Configuración del entrenamiento

Ajusta estos valores según tu hardware. Con `MODO_RAPIDO = True`, todo entrena en pocos minutos incluso en CPU (aunque los resultados serán peores).

In [ ]:
MODO_RAPIDO = os.environ.get("MODO_RAPIDO", "0") == "1"  # pon True si no tienes GPU

N_RECETAS = 600 if MODO_RAPIDO else 5000   # recetas que descargamos
MAX_STEPS = 30 if MODO_RAPIDO else 400     # pasos de entrenamiento por modelo
BATCH_SIZE = 8
OUTPUT_DIR = "outputs"

print(f"MODO_RAPIDO={MODO_RAPIDO} | N_RECETAS={N_RECETAS} | MAX_STEPS={MAX_STEPS}")

### 2.1 Cargar y explorar el dataset

El dataset completo ocupa ~800 MB. Con `streaming=True` descargamos **solo las recetas que necesitamos**. Además, filtramos las recetas demasiado largas para que quepan en la ventana de contexto del entrenamiento.

In [ ]:
from datasets import Dataset, load_dataset

stream = load_dataset("corbt/all-recipes", split="train", streaming=True)
stream = stream.shuffle(seed=SEED, buffer_size=10_000)
stream = stream.filter(lambda x: 200 < len(x["input"]) < 900)

ds_recetas = Dataset.from_list(list(stream.take(N_RECETAS)))
ds_recetas = ds_recetas.train_test_split(test_size=0.1, seed=SEED)
ds_recetas

In [ ]:
print(ds_recetas["train"][0]["input"])
print("=" * 60)
print(ds_recetas["train"][1]["input"])

In [ ]:
largos = [len(t) for t in ds_recetas["train"]["input"]]
plt.hist(largos, bins=40)
plt.title("Longitud de las recetas (caracteres)")
plt.xlabel("caracteres")
plt.show()

### 2.2 Decoder-only: SmolLM2-135M aprende a escribir recetas

**Causal Language Modeling**: el modelo aprende a predecir el siguiente token de cada receta. No hacen falta etiquetas: **el propio texto es la etiqueta** (desplazado una posición). El `DataCollatorForLanguageModeling(mlm=False)` se encarga de crear las etiquetas.

Primero vemos qué genera el modelo **antes** de entrenar:

In [ ]:
nombre = "HuggingFaceTB/SmolLM2-135M"
tok_dec = AutoTokenizer.from_pretrained(nombre)
tok_dec.pad_token = tok_dec.eos_token          # SmolLM2 no tiene token de padding: usamos el de fin de texto
model_dec = AutoModelForCausalLM.from_pretrained(nombre).to(device)

titulos_prueba = ["Chocolate Banana Bread", "Spicy Chicken Tacos", "Grandma's Apple Pie"]


def probar_decoder(model):
    for t in titulos_prueba:
        prompt = f"{t}\n\nIngredients:\n"
        texto = generar(model, tok_dec, prompt, max_new_tokens=120, do_sample=True,
                        temperature=0.7, top_p=0.9, repetition_penalty=1.2)
        print(f"### {prompt}{texto}\n" + "-" * 60)


transformers.set_seed(SEED)
probar_decoder(model_dec)

Tokenizamos el dataset. Añadimos el token de fin de texto (`eos`) al final de cada receta para que el modelo aprenda **cuándo parar**.

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

MAX_LEN_DEC = 256


def tokenizar_dec(batch):
    textos = [t + tok_dec.eos_token for t in batch["input"]]
    return tok_dec(textos, truncation=True, max_length=MAX_LEN_DEC)


ds_dec = ds_recetas.map(tokenizar_dec, batched=True, remove_columns=["input"])
collator_dec = DataCollatorForLanguageModeling(tokenizer=tok_dec, mlm=False)

# Veamos qué produce el collator: input_ids y labels (iguales; el modelo las desplaza internamente).
# Las posiciones de padding se marcan con -100 para que no cuenten en la loss.
ejemplo = collator_dec([ds_dec["train"][0], ds_dec["train"][1]])
print({k: v.shape for k, v in ejemplo.items()})
print(ejemplo["labels"][0][-10:])

In [ ]:
args_dec = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/smollm2-recetas",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=MAX_STEPS // 10,
    weight_decay=0.01,
    logging_steps=max(1, MAX_STEPS // 10),
    eval_strategy="no",
    save_strategy="no",
    bf16=USE_BF16,
    fp16=USE_FP16,
    report_to="none",
    seed=SEED,
)

trainer_dec = Trainer(
    model=model_dec,
    args=args_dec,
    train_dataset=ds_dec["train"],
    eval_dataset=ds_dec["test"],
    data_collator=collator_dec,
    processing_class=tok_dec,
)

# Perplejidad = exp(loss): "entre cuántos tokens duda el modelo de media". Cuanto más baja, mejor.
loss_antes = trainer_dec.evaluate()["eval_loss"]
print(f"ANTES  -> loss: {loss_antes:.3f} | perplejidad: {math.exp(loss_antes):.1f}")

In [ ]:
trainer_dec.train()

loss_despues = trainer_dec.evaluate()["eval_loss"]
print(f"ANTES   -> perplejidad: {math.exp(loss_antes):.1f}")
print(f"DESPUÉS -> perplejidad: {math.exp(loss_despues):.1f}")
if device == "cuda":
    print(f"Pico de VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ¿Ha mejorado el formato de las recetas?
transformers.set_seed(SEED)
probar_decoder(model_dec)

In [ ]:
# Guardar el modelo adaptado (luego se puede cargar con from_pretrained(ruta))
trainer_dec.save_model(f"{OUTPUT_DIR}/smollm2-recetas")
tok_dec.save_pretrained(f"{OUTPUT_DIR}/smollm2-recetas")

del model_dec, trainer_dec
liberar_memoria()

#### ¿Y si el modelo es más grande? LoRA

Con SmolLM2-135M podemos entrenar **todos** los pesos (*full fine-tuning*). Con modelos de miles de millones de parámetros no cabe en memoria, así que se usa **LoRA** (librería `peft`): se congelan los pesos originales y solo se entrenan unas matrices pequeñas añadidas. El resto del código (Trainer, collator...) es **idéntico**.

In [ ]:
from peft import LoraConfig, get_peft_model

model_base = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")
config_lora = LoraConfig(
    r=8,                                    # rango de las matrices LoRA
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],    # capas de atención en las que añadimos LoRA
    task_type="CAUSAL_LM",
)
model_lora = get_peft_model(model_base, config_lora)
model_lora.print_trainable_parameters()     # solo se entrena un porcentaje mínimo de los pesos

del model_base, model_lora
liberar_memoria()

### 2.3 Encoder-decoder: Flan-T5-small, de título a receta

Ahora planteamos el problema como **seq2seq**: la **entrada** es el título y la **salida** son los ingredientes y los pasos. Así que hay que separar el título del resto de la receta.

> **Detalle importante:** el tokenizador de T5 **elimina los saltos de línea** (`\n`). Si entrenamos con el texto tal cual, el modelo nunca podrá generar el formato en líneas. Por eso convertimos cada receta a una sola línea, con `|` como separador, y la volvemos a "maquetar" al mostrarla.

In [ ]:
tok_t5 = AutoTokenizer.from_pretrained("google/flan-t5-small")
print(tok_t5.tokenize("Ingredients:\n- 2 eggs"))  # ¡el \n desaparece!


def separar_receta(texto):
    # 'Título\n\nIngredients:\n- a\n...' -> ('Título', 'Ingredients: | - a | ...')
    titulo, _, cuerpo = texto.partition("\n")
    lineas = [l.strip() for l in cuerpo.strip().split("\n") if l.strip()]
    return titulo.strip(), " | ".join(lineas)


def maquetar(texto_una_linea):
    return texto_una_linea.replace(" | ", "\n")


titulo, cuerpo = separar_receta(ds_recetas["train"][0]["input"])
print("ENTRADA:", titulo)
print("SALIDA :", cuerpo[:200], "...")

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

nombre = "google/flan-t5-small"
tok_t5 = AutoTokenizer.from_pretrained(nombre)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(nombre).to(device)

PREFIJO = "Write a recipe for: "


def probar_t5(model):
    for t in titulos_prueba:
        texto = generar_seq2seq(model, tok_t5, PREFIJO + t, max_new_tokens=200,
                                num_beams=4, no_repeat_ngram_size=3)
        print(f"### {t}\n{maquetar(texto)}\n" + "-" * 60)


probar_t5(model_t5)

In [ ]:
MAX_LEN_ENTRADA, MAX_LEN_SALIDA = 32, 256


def tokenizar_t5(batch):
    titulos, cuerpos = zip(*[separar_receta(t) for t in batch["input"]])
    enc = tok_t5([PREFIJO + t for t in titulos], truncation=True, max_length=MAX_LEN_ENTRADA)
    enc["labels"] = tok_t5(text_target=list(cuerpos), truncation=True, max_length=MAX_LEN_SALIDA)["input_ids"]
    return enc


ds_t5 = ds_recetas.map(tokenizar_t5, batched=True, remove_columns=["input"])
collator_t5 = DataCollatorForSeq2Seq(tokenizer=tok_t5, model=model_t5)
ds_t5

In [ ]:
args_t5 = Seq2SeqTrainingArguments(
    output_dir=f"{OUTPUT_DIR}/flan-t5-recetas",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=5e-4,              # T5 suele necesitar un learning rate más alto que los decoders
    lr_scheduler_type="cosine",
    warmup_steps=MAX_STEPS // 10,
    logging_steps=max(1, MAX_STEPS // 10),
    eval_strategy="no",
    save_strategy="no",
    bf16=USE_BF16,                   # T5 es inestable en fp16 (da NaN): en la T4 entrenamos en fp32
    report_to="none",
    seed=SEED,
)

trainer_t5 = Seq2SeqTrainer(
    model=model_t5,
    args=args_t5,
    train_dataset=ds_t5["train"],
    eval_dataset=ds_t5["test"],
    data_collator=collator_t5,
    processing_class=tok_t5,
)

loss_antes = trainer_t5.evaluate()["eval_loss"]
trainer_t5.train()
loss_despues = trainer_t5.evaluate()["eval_loss"]
print(f"Loss de validación: {loss_antes:.3f} -> {loss_despues:.3f}")

In [ ]:
probar_t5(model_t5)

In [ ]:
trainer_t5.save_model(f"{OUTPUT_DIR}/flan-t5-recetas")
del model_t5, trainer_t5
liberar_memoria()

### 2.4 (Opcional) Encoder: DistilBERT aprende vocabulario de cocina

Un encoder también se puede adaptar al dominio: seguimos con su tarea de preentrenamiento (**Masked LM**) sobre las recetas. Esto se llama *Domain-Adaptive Pretraining* y es el paso previo habitual antes de entrenar un clasificador en un dominio especializado (médico, legal...).

Modelo: [`distilbert-base-uncased`](https://huggingface.co/distilbert/distilbert-base-uncased)

In [ ]:
from transformers import AutoModelForMaskedLM

nombre = "distilbert-base-uncased"
tok_bert = AutoTokenizer.from_pretrained(nombre)
model_bert = AutoModelForMaskedLM.from_pretrained(nombre).to(device)

frases_mask = [
    "Preheat the [MASK] to 350 degrees.",
    "Mix the flour, sugar and [MASK] in a large bowl.",
    "Add 2 cups of shredded [MASK] cheese.",
]


def probar_bert(model):
    fm = pipeline("fill-mask", model=model, tokenizer=tok_bert, device=device)
    for f in frases_mask:
        print(f"{f:50s} -> {[p['token_str'] for p in fm(f, top_k=5)]}")


probar_bert(model_bert)

In [ ]:
ds_bert = ds_recetas.map(lambda b: tok_bert(b["input"], truncation=True, max_length=256),
                         batched=True, remove_columns=["input"])

trainer_bert = Trainer(
    model=model_bert,
    args=TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/distilbert-recetas",
        max_steps=max(10, MAX_STEPS // 2),
        per_device_train_batch_size=16,
        learning_rate=5e-5,
        logging_steps=max(1, MAX_STEPS // 10),
        save_strategy="no",
        bf16=USE_BF16,
        fp16=USE_FP16,
        report_to="none",
        seed=SEED,
    ),
    train_dataset=ds_bert["train"],
    # mlm=True: tapa aleatoriamente el 15 % de los tokens y el modelo debe adivinarlos
    data_collator=DataCollatorForLanguageModeling(tokenizer=tok_bert, mlm=True, mlm_probability=0.15),
)
trainer_bert.train()
probar_bert(model_bert)

In [ ]:
del model_bert, trainer_bert
liberar_memoria()

### Ejercicios — Sección 2

1. Inventa 5 títulos de recetas (algunos raros, como *"Chocolate Pizza with Tuna"*) y compáralos en el decoder y en el T5 adaptados. ¿Cuál respeta mejor el título? ¿Por qué crees que pasa?
2. Repite el entrenamiento del decoder con `MAX_STEPS` a la mitad y al doble. Apunta la perplejidad final y la calidad de las recetas. ¿Hay sobreajuste?
3. Cambia el `learning_rate` del decoder a `5e-4` y a `5e-6`. ¿Qué ocurre con la *loss*?
4. Carga el modelo guardado en `outputs/smollm2-recetas` con `AutoModelForCausalLM.from_pretrained(...)` y genera una receta con un prompt incompleto (por ejemplo, empezando ya los ingredientes).
5. **Reto**: cambia la tarea del T5 para que aprenda **ingredientes → título** (la tarea inversa). Solo tienes que modificar `tokenizar_t5`.
6. **Reto**: entrena el decoder con LoRA (apartado 2.2) en lugar de con *full fine-tuning*. Compara el tiempo, la memoria y el resultado.

---
## 3. Parámetros de generación y *chat templates*

En esta sección trabajamos solo con un **decoder-only** ajustado para chat: [`Qwen/Qwen2.5-0.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct).

### 3.1 ¿Cómo elige el modelo el siguiente token?

En cada paso, el modelo devuelve un **logit** por cada token del vocabulario (más de 150 000 en Qwen). Hay varias estrategias para elegir:

| Parámetro | Qué hace | Efecto |
|---|---|---|
| `do_sample=False` (*greedy*) | Elige siempre el token más probable | Determinista, pero tiende a repetirse |
| `temperature` | Divide los logits por T antes del softmax | T < 1: más conservador · T > 1: más creativo/caótico |
| `top_k` | Solo considera los k tokens más probables | Elimina la "cola" de tokens absurdos |
| `top_p` (*nucleus*) | Solo considera los tokens más probables hasta sumar una probabilidad p | Como top_k, pero adaptativo |
| `repetition_penalty` | Penaliza los tokens que ya han aparecido | Menos bucles |
| `num_beams` | *Beam search*: explora varias secuencias a la vez | Mejores frases "probables", menos variedad |
| `max_new_tokens` | Longitud máxima de la respuesta | – |

In [ ]:
nombre = "Qwen/Qwen2.5-0.5B-Instruct"
tok_chat = AutoTokenizer.from_pretrained(nombre)
model_chat = AutoModelForCausalLM.from_pretrained(nombre, dtype=DTYPE_INFERENCIA).to(device)
print(contar_parametros(model_chat))
print("Tamaño del vocabulario:", len(tok_chat))

Visualicemos la distribución del **siguiente token** y cómo la modifican la temperatura, top-k y top-p:

In [ ]:
prompt = "My favourite ingredient for a pizza is"
enc = tok_chat(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    logits = model_chat(**enc).logits[0, -1].float()

N = 15
top_ids = logits.topk(N).indices
etiquetas = [repr(tok_chat.decode(i)) for i in top_ids]

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, T in zip(axes, [0.5, 1.0, 1.5]):
    probs = (logits / T).softmax(-1)[top_ids].cpu()
    ax.bar(range(N), probs)
    ax.set_xticks(range(N), etiquetas, rotation=70)
    ax.set_title(f"temperature = {T}")
axes[0].set_ylabel("probabilidad")
plt.tight_layout()
plt.show()

In [ ]:
# ¿Qué tokens "sobreviven" a top-k y top-p (con T=1)?
probs = logits.softmax(-1)
ordenadas, indices = probs.sort(descending=True)
acumulada = ordenadas.cumsum(-1)

k, p = 5, 0.9
n_top_p = int((acumulada < p).sum()) + 1
print(f"top_k={k}: se queda con {k} tokens -> {[tok_chat.decode(i) for i in indices[:k]]}")
print(f"top_p={p}: se queda con {n_top_p} tokens (los que suman el {p:.0%} de la probabilidad)")
print("Probabilidad acumulada de los 10 primeros:", [round(x, 3) for x in acumulada[:10].tolist()])

### 3.2 Probando los parámetros

Para no mezclar conceptos, de momento ignora la función `chat`: la explicaremos en el apartado 3.3. Simplemente le pasamos una lista de mensajes y parámetros de generación.

In [ ]:
@torch.no_grad()
def chat(messages, model=None, tok=None, **gen_kwargs):
    model, tok = model or model_chat, tok or tok_chat
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(model.device)
    gen_kwargs.setdefault("max_new_tokens", 80)
    out = model.generate(**inputs, **gen_kwargs)
    return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


pregunta = [{"role": "user", "content": "Write a one-sentence slogan for a bakery."}]

In [ ]:
print("GREEDY (siempre igual):")
for _ in range(2):
    print("  ", chat(pregunta, do_sample=False))

for T in [0.2, 0.8, 1.5]:
    print(f"\ntemperature={T}:")
    for _ in range(3):
        print("  ", chat(pregunta, do_sample=True, temperature=T, top_k=0, top_p=1.0))

In [ ]:
configs = {
    "top_k=1 (equivale a greedy)": dict(do_sample=True, top_k=1),
    "top_k=50": dict(do_sample=True, top_k=50, top_p=1.0, temperature=1.2),
    "top_p=0.5": dict(do_sample=True, top_p=0.5, top_k=0, temperature=1.2),
    "top_p=0.95": dict(do_sample=True, top_p=0.95, top_k=0, temperature=1.2),
    "beam search (4 beams)": dict(do_sample=False, num_beams=4),
}
for nombre_cfg, cfg in configs.items():
    print(f"{nombre_cfg:28s} -> {chat(pregunta, **cfg)}")

In [ ]:
# repetition_penalty: probamos con una tarea que invita a repetirse
lista = [{"role": "user", "content": "List 15 ingredients for a salad, one per line."}]
for rp in [1.0, 1.3]:
    print(f"--- repetition_penalty={rp}")
    print(chat(lista, do_sample=False, repetition_penalty=rp, max_new_tokens=120), "\n")

#### `GenerationConfig`

Cada modelo trae una configuración de generación **por defecto** (decidida por sus autores), que se aplica cuando no indicamos nada. También podemos crear la nuestra y reutilizarla:

In [ ]:
from transformers import GenerationConfig

print("Configuración por defecto de Qwen:\n", model_chat.generation_config)

creativa = GenerationConfig(do_sample=True, temperature=1.1, top_p=0.95, top_k=50,
                            max_new_tokens=60, pad_token_id=tok_chat.eos_token_id)
print("Con GenerationConfig propia:", chat(pregunta, generation_config=creativa))

### 3.3 *Chat templates*

Un modelo de chat **sigue siendo un modelo que predice el siguiente token**: solo ve un único texto. Durante su ajuste se entrenó con conversaciones escritas en un **formato concreto**, con tokens especiales que marcan quién habla. Cada familia de modelos usa un formato distinto.

El *chat template* (guardado en el tokenizador) convierte una lista de mensajes `{"role", "content"}` en ese texto:

In [ ]:
mensajes = [
    {"role": "system", "content": "You are a helpful chef. Answer briefly."},
    {"role": "user", "content": "How long should I boil an egg?"},
]

texto = tok_chat.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
print(texto)

- `<|im_start|>` / `<|im_end|>` son **tokens especiales** que delimitan cada mensaje.
- `add_generation_prompt=True` añade `<|im_start|>assistant\n` al final: le indica al modelo que **ahora le toca responder**.

¿Y si no damos un mensaje *system*? Qwen añade uno por defecto:

In [ ]:
print(tok_chat.apply_chat_template([{"role": "user", "content": "Hi!"}], tokenize=False, add_generation_prompt=True))

In [ ]:
# Generamos con la plantilla (es lo que hace nuestra función `chat`)
print(chat(mensajes, do_sample=False))

In [ ]:
# ¿Qué pasa si NO usamos la plantilla y le pasamos el texto tal cual?
texto_plano = "You are a helpful chef. Answer briefly.\nHow long should I boil an egg?"
print(generar(model_chat, tok_chat, texto_plano, max_new_tokens=80, do_sample=False))

#### Conversaciones de varios turnos

El modelo **no tiene memoria**: en cada llamada hay que enviarle **toda la conversación**. Para continuarla, añadimos la respuesta del asistente y la nueva pregunta del usuario a la lista:

In [ ]:
conversacion = [
    {"role": "system", "content": "You are a helpful chef. Answer briefly."},
    {"role": "user", "content": "Give me a simple pancake recipe."},
]
respuesta = chat(conversacion, do_sample=False, max_new_tokens=150)
print("ASISTENTE:", respuesta)

conversacion += [
    {"role": "assistant", "content": respuesta},
    {"role": "user", "content": "How can I make it without eggs?"},
]
print("\nASISTENTE:", chat(conversacion, do_sample=False, max_new_tokens=150))

Cada familia de modelos usa su propia plantilla. Comparamos con [`SmolLM2-360M-Instruct`](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct), [`TinyLlama-1.1B-Chat-v1.0`](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0) y [`Phi-3.5-mini-instruct`](https://huggingface.co/microsoft/Phi-3.5-mini-instruct).

In [ ]:
# Cada familia de modelos usa su propia plantilla. Solo hace falta descargar el tokenizador, no el modelo.
# SmolLM2 usa el mismo formato que Qwen (llamado "ChatML"); TinyLlama y Phi-3.5 usan otros.
for nombre in ["HuggingFaceTB/SmolLM2-360M-Instruct", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "microsoft/Phi-3.5-mini-instruct"]:
    tok_otro = AutoTokenizer.from_pretrained(nombre)
    print(f"===== {nombre}")
    print(tok_otro.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True))

In [ ]:
del model_chat
liberar_memoria()

### Ejercicios — Sección 3

1. Encuentra una configuración que haga al modelo **muy creativo** y otra que lo haga **totalmente determinista** para la pregunta *"Invent a name for a new dessert"*. Justifica los valores.
2. Pon `temperature=3.0`. ¿Qué pasa? ¿Y si además añades `top_p=0.5`? ¿Por qué top-p "rescata" al modelo?
3. Cambia el mensaje *system* para que el chef responda **siempre en español y como un pirata**. ¿Lo cumple en una conversación de 3 turnos?
4. Rehaz la gráfica del apartado 3.1 con otro prompt. ¿Hay prompts en los que el modelo esté muy seguro (un token con casi toda la probabilidad)?
5. Imprime `tok_chat.chat_template`: es una plantilla **Jinja**. Busca en ella dónde se añade el mensaje *system* por defecto.
6. Mira la plantilla de otro modelo del Hub (por ejemplo, [`meta-llama/Llama-3.2-1B-Instruct`](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) o [`google/gemma-3-1b-it`](https://huggingface.co/google/gemma-3-1b-it), que pueden requerir aceptar su licencia) y compárala con la de Qwen.

---
## 4. ¿Qué aporta el *instruction tuning*?

Un modelo **base** solo ha aprendido a continuar texto (o, en el caso de T5, a hacer unas tareas concretas con prefijos fijos como `translate English to German:`). El **instruction tuning** es un ajuste posterior con miles de ejemplos *(instrucción → respuesta)* de tareas muy variadas. El resultado es un modelo que **entiende instrucciones nuevas** escritas en lenguaje natural.

Google publicó exactamente esto con T5:

- **T5** (2019): preentrenado + multitarea con prefijos fijos.
- **Flan-T5** (2022): el **mismo T5**, ajustado con más de 1 800 tareas en formato de instrucción (colección *FLAN*).

Misma arquitectura y mismo tamaño: **la única diferencia es el instruction tuning**. Vamos a compararlos:

- [`google-t5/t5-base`](https://huggingface.co/google-t5/t5-base)
- [`google/flan-t5-base`](https://huggingface.co/google/flan-t5-base)

In [ ]:
modelos_t5 = {
    "T5-base": "google-t5/t5-base",
    "Flan-T5-base": "google/flan-t5-base",
}

instrucciones = [
    "translate English to German: The kitchen is clean.",   # prefijo "oficial" de T5
    "Is the following review positive or negative? Review: The pasta was cold and the waiter was rude.",
    "Answer the following question. What is the boiling point of water in Celsius?",
    "Translate to Spanish: I like apples.",
    "List three ingredients needed to make pancakes.",
    "Q: I have 5 eggs and I use 2 for a cake and 1 for an omelette. How many eggs are left? "
    "A: Let's think step by step.",
]

resultados = {"Instrucción": instrucciones}
for etiqueta, nombre in modelos_t5.items():
    tok = AutoTokenizer.from_pretrained(nombre)
    model = AutoModelForSeq2SeqLM.from_pretrained(nombre).to(device)
    print(etiqueta, contar_parametros(model))
    resultados[etiqueta] = [generar_seq2seq(model, tok, p, max_new_tokens=60) for p in instrucciones]
    del model
    liberar_memoria()

pd.DataFrame(resultados)

**¿Qué observar?**

- **T5-base** solo sabe hacer las tareas con los **prefijos** con los que se entrenó (`translate English to German:`, `summarize:`...). Ante una instrucción nueva, suele **copiar** parte de la entrada.
- **Flan-T5-base** intenta **responder** a todas las instrucciones, aunque no siempre acierte (es un modelo pequeño).
- Fíjate en la última fila: con *"Let's think step by step"*, Flan-T5 intenta razonar paso a paso (*chain-of-thought*), algo que también aprendió en el instruction tuning.

Lo mismo ocurre con los **decoders**. Comparamos [`SmolLM2-360M`](https://huggingface.co/HuggingFaceTB/SmolLM2-360M) (base) con [`SmolLM2-360M-Instruct`](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct) (el mismo modelo con instruction tuning y alineamiento para chat).

- Al modelo base le pasamos el texto tal cual (no tiene *chat template*).
- Al modelo instruct le pasamos los mensajes con su plantilla.

In [ ]:
preguntas = [
    "What is the capital of Italy?",
    "Give me three tips to cook perfect rice.",
    "Translate to French: Good morning, how are you?",
]

resultados = {"Pregunta": preguntas}

nombre = "HuggingFaceTB/SmolLM2-360M"
tok = AutoTokenizer.from_pretrained(nombre)
model = AutoModelForCausalLM.from_pretrained(nombre, dtype=DTYPE_INFERENCIA).to(device)
resultados["SmolLM2-360M (base)"] = [
    generar(model, tok, p, max_new_tokens=60, do_sample=False, repetition_penalty=1.2) for p in preguntas
]
del model
liberar_memoria()

nombre = "HuggingFaceTB/SmolLM2-360M-Instruct"
tok = AutoTokenizer.from_pretrained(nombre)
model = AutoModelForCausalLM.from_pretrained(nombre, dtype=DTYPE_INFERENCIA).to(device)
resultados["SmolLM2-360M-Instruct"] = [
    chat([{"role": "user", "content": p}], model=model, tok=tok, max_new_tokens=60, do_sample=False)
    for p in preguntas
]
del model
liberar_memoria()

pd.DataFrame(resultados)

### Ejercicios — Sección 4

1. Diseña **5 instrucciones propias** de distintos tipos (clasificación, extracción de información, razonamiento, generación creativa y una "trampa", como una pregunta sin respuesta). Pásalas por los 4 modelos y rellena una tabla con una nota del 0 al 2 para cada respuesta.
2. Con T5-base, prueba los prefijos originales con los que fue entrenado: `summarize:`, `translate English to French:`, `cola sentence:`, `stsb sentence1: ... sentence2: ...`. ¿Funcionan mejor que las instrucciones en lenguaje natural? ¿Por qué?
3. ¿Responde Flan-T5 bien a instrucciones **en español**? ¿Y SmolLM2-Instruct? ¿A qué crees que se debe?
4. Pregunta para reflexionar: en la sección 2, ¿por qué partimos de **Flan**-T5 y no de T5 para adaptarlo a las recetas? ¿Habría alguna diferencia?

---
## 5. Introducción a LangChain

[LangChain](https://python.langchain.com/) es una librería para construir **aplicaciones** con LLMs. No entrena modelos: ofrece piezas intercambiables que se combinan entre sí:

- **Chat models**: una interfaz común para cualquier proveedor (OpenAI, Anthropic, modelos locales...).
- **Prompt templates**: prompts con variables.
- **Output parsers / salida estructurada**: convertir la respuesta en un string, JSON, objetos Pydantic...
- **LCEL** (*LangChain Expression Language*): encadenar piezas con el operador `|`.
- **Tools**: funciones de Python que el LLM puede decidir llamar.
- **Retrievers y vector stores**: buscar documentos relevantes (RAG).

Como LLM usaremos **[OpenRouter](https://openrouter.ai/)**, que da acceso a cientos de modelos (muchos **gratuitos**) con una API compatible con la de OpenAI.

### 5.0 Configuración

1. Crea una cuenta en [openrouter.ai](https://openrouter.ai/) y genera una API key en *Keys*.
2. **En local**: copia `.env.example` a `.env` y pega tu clave: `OPENROUTER_API_KEY=sk-or-...`
3. **En Colab**: abre el panel *Secretos* (icono de la llave), crea `OPENROUTER_API_KEY` y activa el acceso del notebook.

> **Importante:** **nunca** escribas la API key directamente en el notebook ni la subas a GitHub.

In [ ]:
from dotenv import load_dotenv


def obtener_api_key():
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    load_dotenv()
    return os.environ.get("OPENROUTER_API_KEY")


OPENROUTER_API_KEY = obtener_api_key()
assert OPENROUTER_API_KEY, "No se ha encontrado OPENROUTER_API_KEY (revisa tu .env o los Secretos de Colab)"
print("API key cargada")

In [ ]:
from langchain_openai import ChatOpenAI

# "openrouter/free" redirige a un modelo gratuito disponible. También puedes fijar uno concreto
# (los gratuitos acaban en ':free'): https://openrouter.ai/models?max_price=0
MODELO = "openrouter/free"

llm = ChatOpenAI(
    model=MODELO,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0.7,
)

### 5.1 Mensajes: `invoke`, `stream` y `batch`

Igual que con los *chat templates* de la sección 3, la conversación es una **lista de mensajes con roles**. LangChain y el proveedor se encargan de aplicar la plantilla.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

respuesta = llm.invoke([
    SystemMessage(content="You are a Spanish chef. Answer in Spanish, in two sentences."),
    HumanMessage(content="What is the secret of a good tortilla de patatas?"),
])
print(type(respuesta).__name__)
print(respuesta.content)
print("Modelo usado:", respuesta.response_metadata.get("model_name"))
print("Tokens:", respuesta.usage_metadata)

In [ ]:
# stream: recibimos la respuesta token a token (como en ChatGPT)
for trozo in llm.stream("Explica en 3 frases qué es un sofrito."):
    print(trozo.content, end="", flush=True)

In [ ]:
# batch: varias peticiones en paralelo
for r in llm.batch(["Un plato típico de Bilbao, en una línea.", "Un plato típico de Sevilla, en una línea."]):
    print("-", r.content)

### 5.2 *Prompt templates*

Un `ChatPromptTemplate` es una plantilla de mensajes con **variables** entre llaves.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un chef experto en cocina {cocina}. Responde en español de forma breve."),
    ("human", "Propón un plato usando: {ingredientes}"),
])

mensajes = prompt.invoke({"cocina": "japonesa", "ingredientes": "arroz, salmón y aguacate"})
print(mensajes.to_messages())

### 5.3 Cadenas con LCEL: `prompt | llm | parser`

El operador `|` conecta piezas: la salida de una es la entrada de la siguiente. Toda cadena tiene los mismos métodos (`invoke`, `stream`, `batch`).

In [ ]:
from langchain_core.output_parsers import StrOutputParser

cadena = prompt | llm | StrOutputParser()   # StrOutputParser: AIMessage -> str

print(cadena.invoke({"cocina": "italiana", "ingredientes": "tomate, albahaca y queso"}))

In [ ]:
# Encadenar cadenas: (1) inventar una receta en inglés -> (2) traducirla y adaptarla
cadena_receta = (
    ChatPromptTemplate.from_template("Write a very short recipe (max 6 lines) called '{titulo}'.")
    | llm
    | StrOutputParser()
)
cadena_traduccion = (
    ChatPromptTemplate.from_template("Traduce al español esta receta y cambia las unidades al sistema métrico:\n\n{receta}")
    | llm
    | StrOutputParser()
)

cadena_completa = {"receta": cadena_receta} | cadena_traduccion
print(cadena_completa.invoke({"titulo": "Grandma's Apple Pie"}))

### 5.4 Salida estructurada

En las aplicaciones reales no queremos texto libre, sino **datos** que el resto del programa pueda usar. Definimos el esquema con **Pydantic** y LangChain se encarga de que el LLM lo rellene.

In [ ]:
from pydantic import BaseModel, Field


class Receta(BaseModel):
    titulo: str = Field(description="Nombre del plato")
    raciones: int = Field(description="Número de raciones")
    tiempo_minutos: int = Field(description="Tiempo total de preparación en minutos")
    ingredientes: list[str] = Field(description="Lista de ingredientes con cantidades")
    pasos: list[str] = Field(description="Pasos de la preparación")
    vegetariana: bool


llm_estructurado = llm.with_structured_output(Receta, method="function_calling")
receta = llm_estructurado.invoke("Dame una receta sencilla de gazpacho andaluz.")

print(type(receta))
print(receta.titulo, "|", receta.raciones, "raciones |", receta.tiempo_minutos, "min | vegetariana:", receta.vegetariana)
for ing in receta.ingredientes:
    print("  -", ing)

### 5.5 Historial de conversación

Igual que en la sección 3: el LLM **no recuerda nada**. Para conversar, guardamos los mensajes y se los enviamos todos en cada llamada. `MessagesPlaceholder` reserva un hueco en la plantilla para el historial.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

prompt_chat = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente de cocina amable. Responde en español en un máximo de 3 frases."),
    MessagesPlaceholder("historial"),
    ("human", "{pregunta}"),
])
cadena_chat = prompt_chat | llm | StrOutputParser()

historial = []


def conversar(pregunta):
    respuesta = cadena_chat.invoke({"historial": historial, "pregunta": pregunta})
    historial.extend([HumanMessage(content=pregunta), AIMessage(content=respuesta)])
    return respuesta


print("USUARIO: Tengo pollo, limón y patatas.\nASISTENTE:", conversar("Tengo pollo, limón y patatas. ¿Qué cocino?"))
print("\nUSUARIO: ¿Y cuánto tiempo de horno?\nASISTENTE:", conversar("¿Y cuánto tiempo de horno?"))  # solo tiene sentido con el historial

### 5.6 *Tools*: el LLM decide llamar a funciones

Un LLM no sabe calcular con fiabilidad ni consultar datos externos. Con **tools** le describimos funciones de Python (nombre, descripción y argumentos) y **el modelo decide** cuándo llamarlas y con qué argumentos. **Nosotros** ejecutamos la función y le devolvemos el resultado.

In [ ]:
from langchain_core.tools import tool

GRAMOS_POR_TAZA = {"flour": 120, "sugar": 200, "butter": 227, "rice": 185, "milk": 240}


@tool
def tazas_a_gramos(ingrediente: str, tazas: float) -> str:
    # El docstring es la descripción que verá el LLM
    '''Convierte una cantidad en tazas (cups) de un ingrediente a gramos.
    Ingredientes soportados: flour, sugar, butter, rice, milk (en inglés).'''
    g = GRAMOS_POR_TAZA.get(ingrediente.lower())
    if g is None:
        return f"No conozco la densidad de '{ingrediente}'"
    return f"{tazas} tazas de {ingrediente} = {tazas * g:.0f} g"


print(tazas_a_gramos.name, "|", tazas_a_gramos.args)

llm_con_tools = llm.bind_tools([tazas_a_gramos])
pregunta = "¿Cuántos gramos son 2.5 tazas de harina y 1 taza de azúcar?"
ai_msg = llm_con_tools.invoke(pregunta)
print("Llamadas que pide el modelo:", ai_msg.tool_calls)

In [ ]:
# Ejecutamos las tools pedidas y devolvemos los resultados al modelo para que redacte la respuesta final
mensajes = [HumanMessage(content=pregunta), ai_msg]
for llamada in ai_msg.tool_calls:
    resultado = tazas_a_gramos.invoke(llamada)   # devuelve un ToolMessage
    print("TOOL:", resultado.content)
    mensajes.append(resultado)

print("\nASISTENTE:", llm_con_tools.invoke(mensajes).content)

### 5.7 Mini-RAG: preguntar sobre nuestras recetas

**RAG** (*Retrieval-Augmented Generation*): en lugar de confiar en lo que "sabe" el LLM, **buscamos** los documentos relevantes y se los damos en el prompt. Juntamos piezas de todo el notebook:

1. Las **recetas** de la sección 2 son nuestros documentos.
2. Un **encoder** ([`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), sección 1.1.d) calcula los embeddings.
3. Un **vector store** en memoria hace la búsqueda por similitud.
4. El **LLM** responde usando solo esos documentos.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

documentos = [
    Document(page_content=t, metadata={"titulo": t.split("\n")[0]})
    for t in ds_recetas["train"]["input"][:500]
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   model_kwargs={"device": device})
vector_store = InMemoryVectorStore.from_documents(documentos, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

for doc in retriever.invoke("a dessert with chocolate"):
    print("-", doc.metadata["titulo"])

In [ ]:
from langchain_core.runnables import RunnablePassthrough


def formatear(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)


prompt_rag = ChatPromptTemplate.from_template(
    "Responde en español usando SOLO las recetas del contexto. "
    "Si la respuesta no está en el contexto, dilo.\n\n"
    "CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"
)

cadena_rag = (
    {"contexto": retriever | formatear, "pregunta": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

print(cadena_rag.invoke("I want to bake something with chocolate. What do I need?"))

### Ejercicios — Sección 5

1. **Prompt templates**: crea una cadena que reciba `{plato}` y `{nivel}` (principiante/experto) y explique cómo cocinarlo adaptando el lenguaje al nivel.
2. **Cambiar de modelo**: prueba 2 modelos gratuitos distintos de OpenRouter (cambia `MODELO`) con la misma cadena. ¿Notas diferencias de calidad o de velocidad? Mira `response_metadata`.
3. **Salida estructurada**: crea un esquema `ListaCompra` (lista de productos con cantidad y sección del supermercado) y una cadena que, a partir de una `Receta` (apartado 5.4), genere la lista de la compra.
4. **Asistente de cocina**: combina todo en una función `asistente(ingredientes: list[str])` que:
   - a) Use una cadena para proponer una receta estructurada (`Receta`) con esos ingredientes.
   - b) Calcule si es apta para vegetarianos.
   - c) Devuelva un texto final bonito.
5. **Tools**: añade una segunda tool, `escalar_receta(raciones_originales, raciones_nuevas, cantidad)`, y haz una pregunta que obligue al modelo a usar **las dos**.
6. **Memoria**: modifica `conversar` para que solo guarde los **últimos 4 mensajes**. ¿Qué pasa cuando la conversación es larga?
7. **RAG**: añade a la cadena RAG los títulos de las recetas usadas como "fuentes" en la respuesta. Haz una pregunta que **no** esté en las recetas y comprueba si el modelo lo reconoce.
8. **Reflexión**: compara la respuesta del LLM de OpenRouter con la de nuestro Qwen2.5-0.5B local (sección 3) para la misma pregunta. ¿Cuándo tendría sentido usar un modelo pequeño local en vez de una API?

---
## Resumen

- **Encoders** entienden, **decoders** generan y **encoder-decoders** transforman.
- Con `AutoTokenizer` + `AutoModelFor...` cargamos cualquier modelo del Hub; `Trainer` y los *data collators* nos permiten **adaptarlo a un dominio** con pocas líneas de código.
- La **generación** se controla con temperatura, top-k, top-p, penalizaciones y beam search.
- Los modelos de chat necesitan su **chat template**.
- El **instruction tuning** convierte un modelo que "continúa texto" en uno que "sigue instrucciones".
- **LangChain** organiza las aplicaciones con LLMs en piezas componibles: prompts, modelos, parsers, tools y retrievers.